In [16]:
import numpy as np
import pandas as pd


In [17]:
# importando parâmetros
struc_par_path = 'parameters/structural.csv'
inter_par_path = 'parameters/interaction.csv'
structural = pd.read_csv(struc_par_path, sep=';', decimal=',')
interaction = pd.read_csv(inter_par_path, sep=';', decimal=',')

R = structural[['Secondary', 'Volume R']].to_numpy()
Q = structural[['Secondary', 'Surface Area Q']].to_numpy()
A = interaction.iloc[:, 1:].to_numpy()

## Implementação UNIFAC

In [ ]:
def get_species_parameters(species: dict, rk=R, qk=Q) -> dict:
    # Ordenando vetores com base nos grupos secundários
    rk = rk[np.argsort(rk[:, 0])]
    qk = qk[np.argsort(qk[:, 0])]

    # Estrutura de resposta
    parameters = {}
    
    for sp, groups in species.items():
        secondary_groups = np.array([g[1] for g in groups]) # Grupos secundários da espécie i
        groups_qty = np.array([g[2] for g in groups]) # Quantidade do grupo k na espécie i
        id_sec = np.searchsorted(rk[:, 0], secondary_groups) # Índices referentes aos grupos k da espécie i
        r = rk[id_sec][:, 1]
        q = qk[id_sec][:, 1]
        parameters[sp] = [
            groups_qty,
            r,
            q,
        ]

    return parameters

def assemble_interaction_matrix(species, temperature, amn=A):
    # Obtendo todos os grupos principais
    secondary_groups = sorted({
        group[1]
        for molecule in species.values()
        for group in molecule
    })

    # return main_groups
    


def combinatorial_contribution(composition: np.ndarray, group_par: dict, z=10):
    # molec_par = [ri, qi, li, phi_i, theta_i]
    molec_par = np.zeros((len(composition), 5))
    gamma_comb = np.zeros(len(composition))
    for i, pars in enumerate(group_par.values()):
        ri = pars[0] @ pars[1].T  # Calculando ri
        molec_par[i, 0] = ri
        qi = pars[0] @ pars[2].T  # Calculando qi
        molec_par[i, 1] = qi
        molec_par[i, 2] = z/2 * (ri - qi) - (ri - 1) # Calculando li
    
    r_pond = composition @ molec_par[:, 0]
    q_pond = composition @ molec_par[:, 1]
    li_pond = composition @ molec_par[:, 2]
    for i, (xi, (ri, qi)) in enumerate(zip(composition, molec_par[:, 0:2])):
        phi_i = ri * xi / r_pond # Calculando phi_i
        molec_par[i, 3] = phi_i
        theta_i = qi * xi / q_pond # Calculando theta_i
        molec_par[i, 4] = theta_i

        # Calculando contribuição combinatorial
        gamma_comb[i] = (
            np.log(phi_i/xi)
            + (z/2) * qi * np.log(theta_i/phi_i)
            + molec_par[i, 2]
            - (phi_i/xi) * li_pond
            )
    return gamma_comb, molec_par


In [19]:
# Definindo parâmetros
temperatura = 307  # K
composicao = np.array([0.047, 0.953])

# Definindo moléculas e seus grupos
# Espécie1: [
#      [Main Group, Secondary Group, Qty]1,
#      [Main Group, Secondary Group, Qty]2,
#   ]
especies = {
    "1": [
        [1, 1 , 1],
        [9, 18, 1],
    ],
    "2": [
        [1, 1, 2],
        [1, 2, 3],
    ]
}


In [20]:
# Testes
amn = assemble_interaction_matrix(especies, temperatura)
print(amn)

[(1, 1), (2, 1), (18, 9)]
